# CredAudit reproduction

Datasets: DHL-614, GC-1000, LAR-24K. Set `PROJECT_DIR` below, then run all cells. Results go to `PROJECT_DIR/results_v2/`.

In [ ]:
import os, sys, json, time, math, hashlib, pickle, warnings, itertools, platform
from pathlib import Path

PROJECT_DIR = '/content/drive/MyDrive/Research work/Loan IEEE Access/reproduce'
DATA_SUBDIR, RESULTS_SUBDIR = 'data', 'results_v2'
FILES = {'DHL-614': 'train_u6lujuX_CVtuZ9i.csv', 'GC-1000': 'credit.csv', 'LAR-24K': 'loan_data.csv'}

QUICK_MODE = False
SEED = 42
N_JOBS = 2

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    PROJECT_DIR = os.environ.get('CREDAUDIT_DIR', PROJECT_DIR)
QUICK_MODE = bool(int(os.environ.get('CREDAUDIT_QUICK', int(QUICK_MODE))))

PROJECT = Path(PROJECT_DIR)
DATA_DIR = PROJECT / DATA_SUBDIR
RES = PROJECT / (RESULTS_SUBDIR + ('_quick' if QUICK_MODE else ''))
TAB, FIG, CKPT = RES / 'tables', RES / 'figures', RES / 'checkpoints'
for d in (TAB, FIG, CKPT):
    d.mkdir(parents=True, exist_ok=True)

N_SPLITS = 5
N_REPEATS = 1 if QUICK_MODE else 3
N_REPEATS_RESAMPLING = 1 if QUICK_MODE else 6
N_LEAK_SPLITS = 3 if QUICK_MODE else 15
LAR_SUBSAMPLE = 1500 if QUICK_MODE else 6000
N_BOOT = 500 if QUICK_MODE else 5000
print(f'colab={IN_COLAB} quick={QUICK_MODE}\ndata: {DATA_DIR}\nresults: {RES}')

In [ ]:
import os, warnings, platform
import numpy as np, pandas as pd, scipy, sklearn
import scipy.stats as ss
import matplotlib
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold, train_test_split,
                                     cross_val_predict)
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, matthews_corrcoef,
                             balanced_accuracy_score, brier_score_loss, accuracy_score,
                             cohen_kappa_score, recall_score, roc_curve, precision_recall_curve)
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier, HistGradientBoostingClassifier,
                              StackingClassifier)
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.inspection import permutation_importance
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'
pd.set_option('display.width', 200, 'display.max_columns', 30, 'display.max_rows', 200)
print('python', platform.python_version(), '| sklearn', sklearn.__version__, '| numpy', np.__version__,
      '| pandas', pd.__version__, '| scipy', scipy.__version__)

C = dict(blue='#2a78d6', orange='#eb6834', aqua='#1baf7a', yellow='#eda100', violet='#6250d6',
         red='#d03b3b', ink='#0b0b0b', muted='#52514e', grid='#e4e3df')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
                     'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'grid.color': C['grid'], 'grid.linewidth': 0.6, 'axes.axisbelow': True,
                     'axes.edgecolor': C['muted'], 'axes.labelcolor': C['ink'], 'text.color': C['ink'],
                     'xtick.color': C['muted'], 'ytick.color': C['muted'], 'font.size': 10,
                     'axes.titlesize': 11, 'legend.frameon': False, 'lines.linewidth': 2})
DS_COL = {'DHL-614': C['blue'], 'GC-1000': C['orange'], 'LAR-24K': C['aqua']}
SUMMARY = {}

def savefig(name):
    plt.savefig(FIG / f'{name}.png'); plt.show(); plt.close()
    print(f'[saved] figures/{name}.png')

def save_table(df, name, title=None, index=True, floatfmt=4):
    df.to_csv(TAB / f'{name}.csv', index=index)
    print('\n' + '=' * 100 + f'\n{title or name}\n' + '=' * 100)
    print(df.round(floatfmt).to_string(index=index))
    print(f'[saved] tables/{name}.csv')
    return df

def cached(name, fn):
    p = CKPT / f'{name}.pkl'
    if p.exists():
        try:
            with open(p, 'rb') as f:
                r = pickle.load(f)
            print(f'  [resume] {name}')
            return r
        except Exception:
            print(f'  [corrupt checkpoint, recomputing] {name}')
    t0 = time.time(); r = fn()
    tmp = p.with_suffix('.tmp')
    with open(tmp, 'wb') as f:
        pickle.dump(r, f)
    os.replace(tmp, p)
    print(f'  [done {time.time() - t0:6.1f}s] {name}')
    return r

## Data

In [ ]:
def sha256(p):
    return hashlib.sha256(open(p, 'rb').read()).hexdigest()

raw = {k: pd.read_csv(DATA_DIR / f) for k, f in FILES.items()}
LAR_SHA_PAPER = 'f818e574f6a7e4ce3101a5c1ba8ebf404d768d681169e9985b5e45a7565ea97c'
print('LAR-24K checksum matches the manuscript [40]:', sha256(DATA_DIR / FILES['LAR-24K']) == LAR_SHA_PAPER)

DS = {}
d = raw['DHL-614']
DS['DHL-614'] = dict(X=d.drop(columns=['Loan_ID', 'Loan_Status']), y=(d['Loan_Status'] == 'Y').astype(int).values,
                     num=['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History'],
                     cat=['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area'],
                     target="lender's approval decision", pos='approved (Y)')
d = raw['GC-1000']
assert set(d['default'].unique()) == {1, 2}
gnum = ['months_loan_duration', 'amount', 'installment_rate', 'residence_history', 'age', 'existing_credits', 'dependents']
DS['GC-1000'] = dict(X=d.drop(columns=['default']), y=(d['default'] == 1).astype(int).values, num=gnum,
                     cat=[c for c in d.columns if c not in gnum + ['default']],
                     target='realised credit quality', pos='good (default==1)')
d = raw['LAR-24K']
DS['LAR-24K'] = dict(X=d.drop(columns=['Approval']), y=(d['Approval'] == 'Approved').astype(int).values,
                     num=['Income', 'Credit_Score', 'Loan_Amount', 'DTI_Ratio'], cat=['Employment_Status'],
                     target='rule-generated approval flag', pos='Approved')
for k, v in DS.items():
    v['feat'] = v['num'] + v['cat']
    if k == 'LAR-24K':
        idx, _ = train_test_split(np.arange(len(v['y'])), train_size=LAR_SUBSAMPLE, stratify=v['y'], random_state=SEED)
        idx = np.sort(idx)
    else:
        idx = np.arange(len(v['y']))
    v['Xm'], v['ym'] = v['X'].iloc[idx][v['feat']].reset_index(drop=True), v['y'][idx]

rows = []
for k, v in DS.items():
    r = raw[k]
    rows.append(dict(Dataset=k, Rows=len(r), Cols=r.shape[1], Preds=v['X'].shape[1],
                     Num=len(v['num']), Cat=v['X'].shape[1] - len(v['num']),
                     Missing_pct=round(100 * r.isna().sum().sum() / r.size, 2),
                     Positive_rate=round(v['y'].mean(), 4), Target=v['target'], Positive=v['pos'],
                     Rows_modelled=len(v['ym'])))
T2 = save_table(pd.DataFrame(rows), 'table02_dataset_characteristics', 'TABLE 2 - verified dataset characteristics', index=False)
SUMMARY['table2'] = T2.to_dict('records')

## Preprocessing, models, resamplers, metrics

In [ ]:
def make_pre(num, cat):
    try:
        ohe = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(drop='first', handle_unknown='ignore', sparse=False)
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', ohe)]), cat)],
        verbose_feature_names_out=False)

def make_clf(name, weighted=True, seed=SEED):
    cw = 'balanced' if weighted else None
    if name == 'LR': return LogisticRegression(max_iter=2000, class_weight=cw)
    if name == 'LDA': return LinearDiscriminantAnalysis()
    if name == 'GNB': return GaussianNB()
    if name == 'kNN': return KNeighborsClassifier()
    if name == 'SVM-RBF': return SVC(probability=True, class_weight=cw, random_state=seed)
    if name == 'DT': return DecisionTreeClassifier(class_weight=cw, random_state=seed)
    if name == 'RF': return RandomForestClassifier(class_weight=cw, random_state=seed)
    if name == 'ExtraTrees': return ExtraTreesClassifier(class_weight=cw, random_state=seed)
    if name == 'AdaBoost': return AdaBoostClassifier(random_state=seed)
    if name == 'GBM': return GradientBoostingClassifier(random_state=seed)
    if name == 'HistGB': return HistGradientBoostingClassifier(class_weight=cw, random_state=seed)
    if name == 'MLP': return MLPClassifier(max_iter=500, random_state=seed)
    if name == 'Stacking':
        return StackingClassifier([('rf', make_clf('RF', weighted, seed)), ('hgb', make_clf('HistGB', weighted, seed)),
                                   ('svm', make_clf('SVM-RBF', weighted, seed))],
                                  final_estimator=LogisticRegression(max_iter=2000), cv=3)
    raise ValueError(name)

MODELS = ['LR', 'LDA', 'GNB', 'kNN', 'SVM-RBF', 'DT', 'RF', 'ExtraTrees', 'AdaBoost', 'GBM', 'HistGB', 'MLP', 'Stacking']
if QUICK_MODE: MODELS = ['LR', 'LDA', 'GNB', 'kNN', 'DT', 'RF', 'HistGB']

def make_pipe(ds, clf):
    return Pipeline([('pre', make_pre(DS[ds]['num'], DS[ds]['cat'])), ('clf', clf)])

def _minmaj(y):
    c = np.bincount(y, minlength=2); mi = int(np.argmin(c)); return mi, 1 - mi, c

def rs_random_over(X, y, rng):
    mi, ma, c = _minmaj(y); add = rng.choice(np.where(y == mi)[0], c[ma] - c[mi], replace=True)
    return np.vstack([X, X[add]]), np.concatenate([y, y[add]])

def rs_random_under(X, y, rng):
    mi, ma, c = _minmaj(y); keep = rng.choice(np.where(y == ma)[0], c[mi], replace=False)
    idx = np.concatenate([np.where(y == mi)[0], keep]); return X[idx], y[idx]

def _smote_points(Xmin, n_new, rng, k=5, weights=None):
    k = min(k, len(Xmin) - 1)
    if k < 1 or n_new <= 0: return np.empty((0, Xmin.shape[1]))
    nb = NearestNeighbors(n_neighbors=k + 1).fit(Xmin).kneighbors(Xmin, return_distance=False)[:, 1:]
    base = rng.choice(len(Xmin), n_new, p=weights)
    neigh = nb[base, rng.integers(0, k, n_new)]
    return Xmin[base] + rng.random((n_new, 1)) * (Xmin[neigh] - Xmin[base])

def rs_smote(X, y, rng):
    mi, ma, c = _minmaj(y); S = _smote_points(X[y == mi], c[ma] - c[mi], rng)
    return np.vstack([X, S]), np.concatenate([y, np.full(len(S), mi)])

def rs_adasyn(X, y, rng, k=5):
    mi, ma, c = _minmaj(y); Xmin = X[y == mi]
    nb = NearestNeighbors(n_neighbors=min(k + 1, len(X))).fit(X).kneighbors(Xmin, return_distance=False)[:, 1:]
    r = (y[nb] == ma).mean(axis=1)
    w = r / r.sum() if r.sum() > 0 else None
    S = _smote_points(Xmin, c[ma] - c[mi], rng, k, w)
    return np.vstack([X, S]), np.concatenate([y, np.full(len(S), mi)])

def rs_nearmiss(X, y, rng, k=3):
    mi, ma, c = _minmaj(y); Xmin, maj = X[y == mi], np.where(y == ma)[0]
    dist = NearestNeighbors(n_neighbors=min(k, len(Xmin))).fit(Xmin).kneighbors(X[maj])[0].mean(axis=1)
    keep = maj[np.argsort(dist, kind='stable')[:c[mi]]]
    idx = np.concatenate([np.where(y == mi)[0], keep]); return X[idx], y[idx]

def rs_smote_tomek(X, y, rng):
    X, y = rs_smote(X, y, rng)
    nn = NearestNeighbors(n_neighbors=2).fit(X).kneighbors(X, return_distance=False)[:, 1]
    i = np.arange(len(y)); link = (nn[nn] == i) & (y != y[nn])
    return X[~link], y[~link]

RESAMPLERS = {'RandomOver': rs_random_over, 'RandomUnder': rs_random_under, 'SMOTE': rs_smote,
              'ADASYN': rs_adasyn, 'NearMiss': rs_nearmiss, 'SMOTETomek': rs_smote_tomek}

class InFoldResampler(BaseEstimator, ClassifierMixin):
    def __init__(self, strategy='SMOTE', clf=None, seed=SEED):
        self.strategy, self.clf, self.seed = strategy, clf, seed
    def fit_resample(self, X, y):
        return RESAMPLERS[self.strategy](np.asarray(X, float), np.asarray(y), np.random.default_rng(self.seed))
    def fit(self, X, y):
        Xr, yr = self.fit_resample(X, y)
        self.clf_ = clone(self.clf).fit(Xr, yr); self.classes_ = self.clf_.classes_; return self
    def predict_proba(self, X): return self.clf_.predict_proba(X)
    def predict(self, X): return self.clf_.predict(X)

METRICS = ['ROC-AUC', 'PR-AUC', 'F1', 'G-mean', 'MCC', 'Bal.acc', 'Brier']
def all_metrics(y, p, t=0.5):
    yh = (p >= t).astype(int)
    tpr, tnr = recall_score(y, yh, pos_label=1), recall_score(y, yh, pos_label=0)
    return {'ROC-AUC': roc_auc_score(y, p), 'PR-AUC': average_precision_score(y, p), 'F1': f1_score(y, yh),
            'G-mean': math.sqrt(tpr * tnr), 'MCC': matthews_corrcoef(y, yh), 'Bal.acc': (tpr + tnr) / 2,
            'Brier': brier_score_loss(y, p), 'Acc': accuracy_score(y, yh)}

def get_folds(ds, repeats):
    v = DS[ds]
    return list(RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=repeats, random_state=SEED).split(v['Xm'], v['ym']))

def _one_fold(ds, clf, tr, te):
    v = DS[ds]; m = make_pipe(ds, clone(clf)).fit(v['Xm'].iloc[tr], v['ym'][tr])
    return m.predict_proba(v['Xm'].iloc[te])[:, 1]

def cv_probs(ds, clf, repeats):
    folds = get_folds(ds, repeats)
    ps = Parallel(n_jobs=N_JOBS)(delayed(_one_fold)(ds, clf, tr, te) for tr, te in folds)
    return [(te, p) for (_, te), p in zip(folds, ps)]

def nadeau_bengio(a, b, n_train, n_test):
    d = np.asarray(a) - np.asarray(b); n = len(d); var = d.var(ddof=1)
    if var == 0: return 0.0, 1.0
    t = d.mean() / math.sqrt(var * (1 / n + n_test / n_train))
    return t, 2 * ss.t.sf(abs(t), n - 1)

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for i, j in enumerate(o):
        run = max(run, (m - i) * p[j]); adj[j] = min(1, run)
    return adj

def nemenyi_cd(k, n, alpha=0.05):
    return ss.studentized_range.ppf(1 - alpha, k, np.inf) / math.sqrt(2) * math.sqrt(k * (k + 1) / (6 * n))

def boot_ci(x, fn=np.mean, n=None, seed=SEED):
    rng = np.random.default_rng(seed); x = np.asarray(x); n = n or N_BOOT
    b = [fn(x[rng.integers(0, len(x), len(x))]) for _ in range(n)]
    return np.percentile(b, [2.5, 97.5])
print('protocol ready:', len(MODELS), 'classifiers,', len(RESAMPLERS), 'resamplers')

## C1: LAR-24K label rule

In [ ]:
L = raw['LAR-24K'].copy(); yL = DS['LAR-24K']['y']
L['LTI'] = L['Loan_Amount'] / L['Income']
def lar_rule(df, cs=551, dti=50, lti=0.60, drop=None):
    conds = {'employed': df['Employment_Status'] == 'employed', 'credit': df['Credit_Score'] >= cs,
             'dti': df['DTI_Ratio'] <= dti, 'lti': df['LTI'] < lti}
    out = np.ones(len(df), bool)
    for k, c in conds.items():
        if k != drop: out &= c.values
    return out.astype(int)
mism = int((lar_rule(L) != yL).sum())
appr = L[yL == 1]
print(f'Four-term rule mismatches: {mism} / {len(L)}')
print(f'Among approved: min Credit_Score={appr.Credit_Score.min()}, max DTI={appr.DTI_Ratio.max()}, '
      f'max LTI={appr.LTI.max():.5f}, unemployed approved={(appr.Employment_Status != "employed").sum()}')

deltas = {'Credit_Score >= 551': ('cs', 551, (1, 5, 10)), 'DTI_Ratio <= 50': ('dti', 50, (0.5, 2, 5)),
          'Loan/Income < 0.60': ('lti', 0.60, (0.005, 0.02, 0.05))}
rows = {}
for lab, (arg, base, ds_) in deltas.items():
    steps = [-ds_[2], -ds_[1], -ds_[0], 0, ds_[0], ds_[1], ds_[2]]
    rows[lab] = [int((lar_rule(L, **{arg: base + s}) != yL).sum()) for s in steps]
T4 = save_table(pd.DataFrame(rows, index=['-D3', '-D2', '-D1', 'exact', '+D1', '+D2', '+D3']).T,
                'table04_threshold_perturbation', 'TABLE 4 - mismatches (of 24,000) when each boundary is displaced')
abl = {k: int((lar_rule(L, drop=k) != yL).sum()) for k in ['employed', 'credit', 'lti', 'dti']}
save_table(pd.Series(abl, name='mismatches').to_frame(), 'table04b_condition_ablation', 'Condition ablation - mismatches when a conjunct is removed')

Xt = pd.get_dummies(L[['Income', 'Credit_Score', 'Loan_Amount', 'DTI_Ratio', 'Employment_Status', 'LTI']], drop_first=True).astype(float)
tr, te = train_test_split(np.arange(len(L)), test_size=0.3, stratify=yL, random_state=SEED)
acc = {}
for lab, cols in [('with LTI', list(Xt.columns)), ('without LTI', [c for c in Xt.columns if c != 'LTI'])]:
    t_ = DecisionTreeClassifier(max_depth=4, random_state=SEED).fit(Xt.iloc[tr][cols], yL[tr])
    acc[lab] = accuracy_score(yL[te], t_.predict(Xt.iloc[te][cols]))
    if lab == 'with LTI': tree_lti, cols_lti = t_, cols
print(f"Depth-4 tree hold-out accuracy: with LTI = {acc['with LTI']:.4f}, without LTI = {acc['without LTI']:.4f}")
plt.figure(figsize=(11, 6))
plot_tree(tree_lti, feature_names=cols_lti, class_names=['Rejected', 'Approved'], filled=True, proportion=True, impurity=False, fontsize=8)
plt.title(f"Fig. 2 - depth-4 tree recovering the LAR-24K label rule (hold-out accuracy {acc['with LTI']:.2%})")
savefig('fig02_lar_rule_tree')

ct = pd.crosstab(L['Text'], yL); chi2, p_text = ss.chi2_contingency(ct)[:2]
print(f'Free-text: {L.Text.nunique()} unique templates over {len(L)} rows; chi-square vs label p = {p_text:.3f}')
SUMMARY['C1_rule'] = dict(mismatches=mism, tree_acc_with_lti=acc['with LTI'], tree_acc_without_lti=acc['without LTI'],
                          ablation=abl, text_templates=int(L.Text.nunique()), text_chi2_p=float(p_text))

## C1: oversampling before vs after the split

In [ ]:
def row_hash(X, y):
    return pd.util.hash_pandas_object(X.assign(_y=y).astype(str), index=False).values

def leak_split(ds, s):
    v = DS[ds]; X, y = v['X'][v['feat']].reset_index(drop=True), v['y']; rng = np.random.default_rng(s)
    rf = lambda: make_pipe(ds, RandomForestClassifier(random_state=s, n_jobs=1))
    tr, te = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=s)
    mi, ma, c = _minmaj(y[tr]); add = rng.choice(tr[y[tr] == mi], c[ma] - c[mi], replace=True); tr2 = np.concatenate([tr, add])
    m = rf().fit(X.iloc[tr2], y[tr2]); p = m.predict_proba(X.iloc[te])[:, 1]
    clean = dict(acc=accuracy_score(y[te], p >= .5), auc=roc_auc_score(y[te], p))
    mi, ma, c = _minmaj(y); allidx = np.concatenate([np.arange(len(y)), rng.choice(np.where(y == mi)[0], c[ma] - c[mi], replace=True)])
    Xo, yo = X.iloc[allidx].reset_index(drop=True), y[allidx]
    tr, te = train_test_split(np.arange(len(yo)), test_size=0.2, stratify=yo, random_state=s)
    m = rf().fit(Xo.iloc[tr], yo[tr]); p = m.predict_proba(Xo.iloc[te])[:, 1]; yh = (p >= .5).astype(int)
    dup = np.isin(row_hash(Xo.iloc[te], yo[te]), row_hash(Xo.iloc[tr], yo[tr])).mean()
    return dict(seed=s, acc_clean=clean['acc'], auc_clean=clean['auc'], acc_leaky=accuracy_score(yo[te], yh),
                auc_leaky=roc_auc_score(yo[te], p), kappa_leaky=cohen_kappa_score(yo[te], yh), dup=dup)

leak = {}
for ds in DS:
    leak[ds] = cached(f'leak_{ds}', lambda: pd.DataFrame(Parallel(n_jobs=N_JOBS)(delayed(leak_split)(ds, s) for s in range(N_LEAK_SPLITS))))
rows = []
for ds, df in leak.items():
    infl = 100 * (df.acc_leaky - df.acc_clean)
    rows.append({'Dataset': ds, 'Splits': len(df), 'Leak-free acc': df.acc_clean.mean(), 'Leaky acc': df.acc_leaky.mean(),
                 'Inflation pp (mean)': infl.mean(), 'Inflation pp (sd)': infl.std(ddof=1), 'min pp': infl.min(), 'max pp': infl.max(),
                 'Duplicated test rows %': 100 * df.dup.mean(), 'Leak-free AUC': df.auc_clean.mean(), 'Leaky AUC': df.auc_leaky.mean(),
                 'max |kappa-(2acc-1)| leaky': (df.kappa_leaky - (2 * df.acc_leaky - 1)).abs().max()})
T5 = save_table(pd.DataFrame(rows), 'table05_resample_before_split', 'TABLE 5 - resampling before vs after the split', index=False)
SUMMARY['C1_leak'] = T5.set_index('Dataset').to_dict('index')

fig, ax = plt.subplots(1, 2, figsize=(11, 4)); x = np.arange(len(DS)); w = 0.36
ax[0].bar(x - w / 2 - .01, 100 * T5['Leak-free acc'], w, color=C['blue'], label='Leak-free (resample in training part)')
ax[0].bar(x + w / 2 + .01, 100 * T5['Leaky acc'], w, color=C['red'], label='Resample before split')
for i, r in T5.iterrows():
    ax[0].text(i + w / 2, 100 * r['Leaky acc'] + .6, f"+{r['Inflation pp (mean)']:.1f} pp", ha='center', fontsize=9)
ax[0].set_xticks(x, T5.Dataset); ax[0].set_ylim(60, 108); ax[0].set_ylabel('Accuracy (%)'); ax[0].legend(loc='upper left')
ax[0].set_title(f'(a) Mean accuracy over {N_LEAK_SPLITS} splits')
ax[1].bar(x, T5['Inflation pp (mean)'], 0.5, color=C['muted'],
          yerr=[T5['Inflation pp (mean)'] - T5['min pp'], T5['max pp'] - T5['Inflation pp (mean)']], capsize=5, error_kw=dict(lw=1.2))
for i, r in T5.iterrows():
    ax[1].text(i, r['max pp'] + 1.0, f"{r['Duplicated test rows %']:.0f}% dup.", ha='center', fontsize=9)
ax[1].set_xticks(x, T5.Dataset); ax[1].set_ylabel('Accuracy inflation (pp)'); ax[1].set_title('(b) Inflation: mean, min and max')
savefig('fig03_leak_inflation')

## C1: trivial baselines and cross-dataset transfer

In [ ]:
def best_one_rule(X, y):
    maj = max(y.mean(), 1 - y.mean()); best = (maj, 'majority class', None)
    for c in X.columns:
        s = X[c]
        if pd.api.types.is_numeric_dtype(s) and s.nunique() > 10:
            ok = s.notna().values; st = DecisionTreeClassifier(max_depth=1).fit(s[ok].values.reshape(-1, 1), y[ok])
            pred = np.full(len(y), int(round(y.mean()))); pred[ok] = st.predict(s[ok].values.reshape(-1, 1))
            desc = f'{c} threshold {st.tree_.threshold[0]:.4g}'
        else:
            lev = s.astype(object).where(s.notna(), 'MISSING').astype(str); m = pd.Series(y).groupby(lev.values).mean() >= 0.5
            pred = lev.map(m).astype(int).values
            desc = f'{c}: positive iff level in {sorted(m[m].index.tolist())}' if not m.all() else f'{c}: all levels positive'
        a = (pred == y).mean()
        if a > best[0] + 1e-12: best = (a, desc, c)
    return maj, best
rows = []
for ds, v in DS.items():
    maj, (a, desc, col) = best_one_rule(v['X'][v['feat']], v['y'])
    rows.append({'Dataset': ds, 'Majority': maj, 'Best 1-rule acc': a, 'Rule': desc})
T6 = save_table(pd.DataFrame(rows), 'table06_trivial_baselines', 'TABLE 6 - trivial baselines (model columns are added in Section 9)', index=False)
print('LAR-24K four-term rule accuracy:', (lar_rule(L) == yL).mean())

names = {k: set(v['X'].columns) for k, v in DS.items()}
shared = {f'{a} & {b}': sorted(names[a] & names[b]) for a, b in itertools.combinations(DS, 2)}
print('\nShared predictor names:', shared)
def transfer():
    lar, gc = DS['LAR-24K'], DS['GC-1000']; rng = np.random.default_rng(SEED)
    m = HistGradientBoostingClassifier(random_state=SEED).fit(lar['X'][lar['num']].values, lar['y'])
    pos = roc_auc_score(gc['y'], m.predict_proba(gc['X'][gc['num'][:4]].values)[:, 1])
    perm = [roc_auc_score(gc['y'], m.predict_proba(gc['X'][list(rng.permutation(gc['num'])[:4])].values)[:, 1]) for _ in range(20)]
    nat = [roc_auc_score(gc['ym'][te], p) for te, p in cv_probs('GC-1000', HistGradientBoostingClassifier(random_state=SEED), N_REPEATS)]
    return dict(position_aligned_auc=pos, random_alignment_mean=float(np.mean(perm)), random_alignment_sd=float(np.std(perm)),
                native_gc_auc=float(np.mean(nat)))
tr_res = cached('transfer', transfer)
save_table(pd.Series(tr_res, name='ROC-AUC').to_frame(), 'table_transfer', 'Cross-dataset "transfer" LAR-24K -> GC-1000 (arbitrary feature alignment)')
SUMMARY['C1_baselines'] = T6.set_index('Dataset').to_dict('index'); SUMMARY['C1_transfer'] = dict(shared=shared, **tr_res)

## C2: kappa identity and detectors D1-D7

In [ ]:
def kappa_from(pi, q, a):
    pe = pi * q + (1 - pi) * (1 - q); return (a - pe) / (1 - pe), pe
rng = np.random.default_rng(SEED); maxerr = 0; n_cfg = 0
while n_cfg < 9064:
    pi, tpr, tnr = rng.uniform(.05, .95), rng.uniform(0, 1), rng.uniform(0, 1)
    a = pi * tpr + (1 - pi) * tnr; q = pi * tpr + (1 - pi) * (1 - tnr); k, pe = kappa_from(pi, q, a)
    if abs(1 - pe) < 1e-6: continue
    maxerr = max(maxerr, abs((k - (2 * a - 1)) - (1 - a) * (1 - 2 * pe) / (1 - pe))); n_cfg += 1
print(f'Proposition 1 closed form vs direct computation over {n_cfg} configurations: max |error| = {maxerr:.1e}')

def d1_label_determinism(X, y, num, cat, max_depth=4, thr=0.995, seed=SEED):
    Z = pd.get_dummies(X[num + cat], columns=cat, drop_first=True).astype(float)
    for a, b in itertools.permutations(num, 2):
        Z[f'({a}/{b})'] = (X[a] / X[b].replace(0, np.nan))
    Z = Z.replace([np.inf, -np.inf], np.nan); Z = Z.fillna(Z.median()).fillna(0)
    tr, te = train_test_split(np.arange(len(y)), test_size=0.3, stratify=y, random_state=seed)
    t = DecisionTreeClassifier(max_depth=max_depth, random_state=seed).fit(Z.iloc[tr], y[tr])
    acc = accuracy_score(y[te], t.predict(Z.iloc[te])); rule = None
    if acc >= thr:
        tt = t.tree_; leaves = [i for i in range(tt.node_count) if tt.children_left[i] == -1 and tt.value[i][0].argmax() == 1]
        leaf = max(leaves, key=lambda i: tt.n_node_samples[i]); parent = {}
        for i in range(tt.node_count):
            if tt.children_left[i] != -1:
                parent[tt.children_left[i]] = (i, '<='); parent[tt.children_right[i]] = (i, '>')
        terms, n = [], leaf
        while n in parent:
            p, op = parent[n]; terms.append(f'{Z.columns[tt.feature[p]]} {op} {tt.threshold[p]:.4g}'); n = p
        rule = ' AND '.join(reversed(terms))
    return dict(check='D1', severity='critical' if acc >= thr else 'ok', holdout_acc=acc, rule=rule)

def d2_partition_overlap(Xtr, ytr, Xte, yte):
    f = float(np.isin(row_hash(Xte, yte), row_hash(Xtr, ytr)).mean())
    return dict(check='D2', severity='critical' if f > 0 else 'ok', dup_fraction=f)

def d2b_interpolation(Xtr, ytr, Xte, yte, k=10, tol=1e-6):
    Xtr, Xte, ytr, yte = np.asarray(Xtr, float), np.asarray(Xte, float), np.asarray(ytr), np.asarray(yte)
    hit = np.zeros(len(Xte), bool)
    for c in np.unique(ytr):
        A = Xtr[ytr == c]; ti = np.where(yte == c)[0]
        if len(A) < 3 or len(ti) == 0: continue
        kk = min(k, len(A)); dist, nb = NearestNeighbors(n_neighbors=kk).fit(A).kneighbors(Xte[ti]); P = A[nb]
        for i in range(kk):
            for j in range(i + 1, kk):
                dab = np.linalg.norm(P[:, i] - P[:, j], axis=1)
                ok = (dist[:, i] > 1e-9) & (dist[:, j] > 1e-9) & (dab > 1e-9) & (dist[:, i] + dist[:, j] - dab <= tol * dab)
                hit[ti[ok]] = True
    f = float(hit.mean())
    return dict(check='D2b', severity='critical' if f > 0.01 else 'ok', interpolated_fraction=f)

def d3_pipeline_order(est):
    def walk(o, enclosed):
        found = []
        is_sampler = hasattr(o, 'fit_resample')
        safe_wrapper = isinstance(o, (Pipeline, InFoldResampler))
        if is_sampler and not (enclosed or isinstance(o, InFoldResampler)): found.append(type(o).__name__)
        if hasattr(o, 'get_params'):
            for v in o.get_params(deep=False).values():
                for c in (v if isinstance(v, (list, tuple)) else [v]):
                    c = c[1] if isinstance(c, tuple) and len(c) >= 2 else c
                    if hasattr(c, 'get_params') or hasattr(c, 'fit_resample'): found += walk(c, enclosed or safe_wrapper)
        return found
    bad = walk(est, False)
    return dict(check='D3', severity='critical' if bad else 'ok', bare_samplers=bad)

def d4_trivial_baseline(X, y, reported_acc=None):
    maj, (a, desc, _) = best_one_rule(X, y)
    sev = 'warn' if reported_acc is None else ('critical' if reported_acc <= a else 'ok')
    return dict(check='D4', severity=sev, majority=maj, best_rule_acc=a, rule=desc)

def d5_kappa_consistency(acc, kappa, tol=0.025, frac=0.8):
    dev = np.abs(np.asarray(kappa) - (2 * np.asarray(acc) - 1)); ok = dev <= tol
    return dict(check='D5', severity='critical' if ok.mean() >= frac else 'ok', rows_satisfying=int(ok.sum()), rows=len(ok), max_dev=float(dev.max()))

def d6_power(n_folds, n_contrasts, alpha=0.05):
    pmin = min(1.0, n_contrasts * 2 / 2 ** n_folds)
    return dict(check='D6', severity='critical' if pmin >= alpha else 'ok', min_adjusted_p=pmin,
                min_folds_needed=int(math.ceil(1 + math.log2(n_contrasts / alpha))))

def d7_schema_overlap(schemas):
    pairs = {f'{a} & {b}': len(set(schemas[a]) & set(schemas[b])) for a, b in itertools.combinations(schemas, 2)}
    return dict(check='D7', severity='critical' if min(pairs.values()) == 0 else 'ok', shared=pairs)

def audit_dataset(name):
    v = DS[name]; X = v['X'][v['feat']].reset_index(drop=True); y = v['y']
    out = [d1_label_determinism(X, y, v['num'], v['cat'])]
    f = [d2_partition_overlap(X.iloc[tr], y[tr], X.iloc[te], y[te])['dup_fraction']
         for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y)]
    out.append(dict(check='D2', severity='critical' if max(f) > 0 else 'ok', dup_fraction=float(np.mean(f))))
    g = []
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y):
        pre = make_pre(v['num'], v['cat']).fit(X.iloc[tr])
        g.append(d2b_interpolation(pre.transform(X.iloc[tr]), y[tr], pre.transform(X.iloc[te]), y[te])['interpolated_fraction'])
    out.append(dict(check='D2b', severity='critical' if max(g) > 0.01 else 'ok', interpolated_fraction=float(np.mean(g))))
    out.append(d4_trivial_baseline(X, y)); return out

rows = []
for ds in DS:
    for r in cached(f'audit_v2_{ds}', lambda: audit_dataset(ds)):
        rows.append({'Check': r['check'], 'Input': ds, 'Severity': r['severity'],
                     'Detail': '; '.join(f'{k}={v:.4f}' if isinstance(v, float) else f'{k}={v}' for k, v in r.items() if k not in ('check', 'severity'))})
r = d3_pipeline_order(make_pipe('GC-1000', make_clf('RF'))); rows.append({'Check': 'D3', 'Input': 'benchmark pipeline (no sampler)', 'Severity': r['severity'], 'Detail': str(r['bare_samplers'])})
r = d3_pipeline_order(make_pipe('GC-1000', InFoldResampler('SMOTE', make_clf('RF')))); rows.append({'Check': 'D3', 'Input': 'sampler enclosed in pipeline', 'Severity': r['severity'], 'Detail': str(r['bare_samplers'])})
class BareSampler:
    def fit_resample(self, X, y): return X, y
r = d3_pipeline_order(BareSampler()); rows.append({'Check': 'D3', 'Input': 'bare sampler', 'Severity': r['severity'], 'Detail': str(r['bare_samplers'])})
for nf in (10, 30):
    r = d6_power(nf, 108); rows.append({'Check': 'D6', 'Input': f'{nf} folds x 108 contrasts', 'Severity': r['severity'],
                                        'Detail': f"min Holm-adjusted p={r['min_adjusted_p']:.2e}; folds needed={r['min_folds_needed']}"})
r = d7_schema_overlap({k: v['X'].columns for k, v in DS.items()}); rows.append({'Check': 'D7', 'Input': 'three study schemas', 'Severity': r['severity'], 'Detail': str(r['shared'])})
for ds, df in leak.items():
    r = d5_kappa_consistency(df.acc_leaky.round(2), df.kappa_leaky.round(2))
    rows.append({'Check': 'D5', 'Input': f'{ds}: leaky-arm table ({len(df)} rows)', 'Severity': r['severity'], 'Detail': f"{r['rows_satisfying']}/{r['rows']} rows satisfy kappa=2a-1; max dev {r['max_dev']:.3f}"})
T9 = save_table(pd.DataFrame(rows), 'table09_seven_checks', 'TABLE 9 - the seven checks applied', index=False)
SUMMARY['C2_checks'] = T9.to_dict('records')

In [ ]:
def d5_simulation(n_tables=400 if not QUICK_MODE else 60, rows_per=11, n_test=200):
    rng = np.random.default_rng(SEED); out = []
    for pi in [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]:
        flags = 0
        for _ in range(n_tables):
            acc, kap = [], []
            for _ in range(rows_per):
                npos = int(round(pi * n_test)); y = np.r_[np.ones(npos), np.zeros(n_test - npos)]
                tpr, tnr = rng.uniform(.55, .98), rng.uniform(.55, .98)
                yh = np.where(y == 1, rng.random(n_test) < tpr, rng.random(n_test) > tnr).astype(int)
                acc.append(round(accuracy_score(y, yh), 2)); kap.append(round(cohen_kappa_score(y, yh), 2))
            flags += d5_kappa_consistency(acc, kap)['severity'] == 'critical'
        out.append(dict(pi=pi, flag_rate=flags / n_tables))
    return pd.DataFrame(out)

def d2_injection():
    v = DS['GC-1000']; X, y = v['Xm'], v['ym']; rng = np.random.default_rng(SEED); out = []
    tr, te = train_test_split(np.arange(len(y)), test_size=0.3, stratify=y, random_state=SEED)
    m = make_pipe('GC-1000', HistGradientBoostingClassifier(random_state=SEED)).fit(X.iloc[tr], y[tr])
    for frac in [0, .05, .10, .20, .30, .40, .50]:
        k = int(round(frac * len(te))); src = rng.choice(tr, k, replace=False)
        Xte = pd.concat([X.iloc[src], X.iloc[te[k:]]]); yte = np.r_[y[src], y[te[k:]]]
        out.append(dict(injected=100 * k / len(te), detected=100 * d2_partition_overlap(X.iloc[tr], y[tr], Xte, yte)['dup_fraction'],
                        reported_acc=100 * accuracy_score(yte, m.predict(Xte))))
    return pd.DataFrame(out)

def d2_scaling():
    v = DS['GC-1000']; out = []
    for mult in [1, 4, 8, 24, 72]:
        X = pd.concat([v['Xm']] * mult, ignore_index=True); y = np.tile(v['ym'], mult); h = len(y) // 2; best = 1e9
        for _ in range(3):
            t0 = time.perf_counter(); d2_partition_overlap(X.iloc[:h], y[:h], X.iloc[h:], y[h:]); best = min(best, time.perf_counter() - t0)
        out.append(dict(rows=len(y), ms=1000 * best))
    return pd.DataFrame(out)
sim5, inj2, sc2 = cached('d5_sim', d5_simulation), cached('d2_inj', d2_injection), cached('d2_scale', d2_scaling)
save_table(sim5, 'fig05a_d5_flag_rate', 'D5 - rate of "critical" findings vs true positive rate of the scored partition', index=False)
save_table(inj2, 'fig05b_d2_injection', 'D2 - controlled leakage injection on GC-1000', index=False, floatfmt=2)
save_table(sc2, 'fig05c_d2_scaling', 'D2 - runtime scaling', index=False, floatfmt=1)
fig, ax = plt.subplots(1, 4, figsize=(15, 3.6))
ax[0].plot(sim5.pi, sim5.flag_rate, 'o-', color=C['blue'])
for ds, v in DS.items():
    b = max(v['y'].mean(), 1 - v['y'].mean()); ax[0].axvline(b, color=DS_COL[ds], ls='--', lw=1.2, label=f'{ds} base rate')
ax[0].set(xlabel='True positive rate of scored partition', ylabel='D5 flags "critical"', title='(a) D5 sensitivity / specificity'); ax[0].legend(fontsize=7, loc='center right')
ax[1].plot(inj2.injected, inj2.injected, ':', color=C['muted'], label='injected (truth)'); ax[1].plot(inj2.injected, inj2.detected, 'o', color=C['blue'], label='D2 detected')
ax[1].set(xlabel='Leakage injected (% of test rows)', ylabel='Detected (%)', title='(b) D2 under controlled injection'); ax[1].legend(fontsize=8)
ax[2].plot(inj2.injected, inj2.reported_acc, 's-', color=C['red'])
ax[2].set(xlabel='Leakage injected (% of test rows)', ylabel='Reported accuracy (%)', title='(c) Accuracy inflation from injection')
ax[3].plot(sc2.rows, sc2.ms, 'o-', color=C['blue']); ax[3].set(xlabel='Rows', ylabel='D2 runtime (ms)', title='(d) Runtime scaling')
plt.tight_layout(); savefig('fig05_detector_evaluation')
SUMMARY['C2_eval'] = dict(d5=sim5.to_dict('records'), d2_injection=inj2.to_dict('records'), d2_scaling=sc2.to_dict('records'), prop1_max_err=maxerr)

## C3: benchmark

In [ ]:
bench = {}
oof = {}
for ds in DS:
    print(ds)
    for mname in MODELS:
        res = cached(f'bench_{ds}_{mname}', lambda: cv_probs(ds, make_clf(mname), N_REPEATS))
        oof[(ds, mname)] = res
        bench[(ds, mname)] = pd.DataFrame([all_metrics(DS[ds]['ym'][te], p) for te, p in res])
NF = N_SPLITS * N_REPEATS
auc = {ds: pd.DataFrame({m: bench[(ds, m)]['ROC-AUC'].values for m in MODELS}) for ds in DS}
rank = {ds: auc[ds].rank(axis=1, ascending=False) for ds in DS}

T10 = pd.DataFrame({ds: auc[ds].mean().map('{:.4f}'.format) + ' ± ' + auc[ds].std(ddof=1).map('{:.4f}'.format) for ds in DS})
for i, ds in enumerate(DS): T10[f'r{i + 1}'] = rank[ds].mean().round(1)
T10['_s'] = auc['GC-1000'].mean(); T10 = T10.sort_values('_s', ascending=False).drop(columns='_s')
save_table(T10, 'table10_roc_auc', f'TABLE 10 - ROC-AUC mean ± sd over {NF} folds; r1–r3 = mean Friedman rank per dataset')
for ds, tn in [('GC-1000', 'table16_full_metrics_gc1000'), ('DHL-614', 'table17_full_metrics_dhl614'), ('LAR-24K', 'tableS_full_metrics_lar24k')]:
    full = pd.DataFrame({m: bench[(ds, m)][METRICS].mean() for m in MODELS}).T.sort_values('ROC-AUC', ascending=False)
    save_table(full, tn, f'Full metric panel - {ds} (means over {NF} folds, threshold 0.5)')
pd.concat([bench[k].assign(dataset=k[0], model=k[1], fold=range(len(bench[k]))) for k in bench]).to_csv(TAB / 'per_fold_results_benchmark.csv', index=False)
print('[saved] tables/per_fold_results_benchmark.csv')

## C3: statistical comparison, ROC/PR

In [ ]:
rows, holm_tabs = [], {}
k = len(MODELS); cd = nemenyi_cd(k, NF)
for ds in DS:
    A = auc[ds]; chi, p = ss.friedmanchisquare(*[A[m] for m in MODELS])
    order = A.mean().sort_values(ascending=False); best, second = order.index[:2]
    n = len(DS[ds]['ym']); t, pnb = nadeau_bengio(A[best], A[second], n * (N_SPLITS - 1) / N_SPLITS, n / N_SPLITS)
    praw = []
    for m in order.index[1:]:
        dlt = A[best] - A[m]; praw.append(1.0 if (dlt == 0).all() else ss.wilcoxon(A[best], A[m]).pvalue)
    nbs = [nadeau_bengio(A[best], A[m], n * .8, n * .2)[1] for m in order.index[1:]]
    holm_tabs[ds] = pd.DataFrame({'dAUC vs best': (order[best] - order[1:]).values, 'Wilcoxon p': praw, 'Holm p': holm(praw),
                                  'Nadeau-Bengio p': nbs, 'NB Holm p': holm(nbs)}, index=order.index[1:])
    tied_nem = int(((rank[ds].mean() - rank[ds].mean().min()) < cd).sum())
    rows.append({'Dataset': ds, 'k/N': f'{k}/{NF}', 'Friedman chi2': chi, 'p': p, 'CD(0.05)': cd, 'Best': best, 'Best AUC': order[best],
                 'Runner-up': second, 'dAUC': order[best] - order[second], 'NB t': t, 'NB p': pnb,
                 'models within CD of best (Nemenyi)': tied_nem, 'models NOT separable from best (NB, Holm)': int((holm(nbs) >= .05).sum()) + 1})
    save_table(holm_tabs[ds], f'table11b_best_vs_rest_{ds}', f'{ds}: best model ({best}) against each competitor', floatfmt=5)
T11 = pd.DataFrame(rows)
T11['p'] = T11['p'].map('{:.2e}'.format)
save_table(T11, 'table11_statistical_comparison', 'TABLE 11 - omnibus test vs corrected best-vs-runner-up test', index=False)
MR = pd.DataFrame({ds: auc[ds].mean().rank(ascending=False) for ds in DS}); chi_d, p_d = ss.friedmanchisquare(*[MR.loc[m] for m in MODELS])
cd3 = nemenyi_cd(k, len(DS)); spread = MR.mean(axis=1).max() - MR.mean(axis=1).min()
print(f'\nFriedman across datasets (Demsar): chi2({k - 1}) = {chi_d:.1f}, p = {p_d:.3f}; Nemenyi CD (N=3) = {cd3:.1f}; largest mean-rank difference = {spread:.1f}')
SUMMARY['C3'] = dict(table11=T11.to_dict('records'), friedman_across=dict(chi2=chi_d, p=p_d, cd=cd3, max_rank_diff=spread),
                     mean_auc={ds: auc[ds].mean().to_dict() for ds in DS})
T6b = T6.copy(); T6b['Best model (AUC)'] = [f"{r['Best']} ({r['Best AUC']:.3f})" for r in rows]
T6b['Best model bal.acc'] = [bench[(r['Dataset'], r['Best'])]['Bal.acc'].mean() for r in rows]
T6b['Best model acc @0.5'] = [bench[(r['Dataset'], r['Best'])]['Acc'].mean() for r in rows]
T6b['Max acc @0.5 over all 13 models'] = [max(bench[(ds, m)]['Acc'].mean() for m in MODELS) for ds in DS]
save_table(T6b, 'table06_trivial_baselines', 'TABLE 6 - trivial baselines against the benchmark models', index=False)
SUMMARY['C1_baselines'] = T6b.set_index('Dataset').to_dict('index')

fig, axes = plt.subplots(len(DS), 1, figsize=(10, 3.4 * len(DS)))
for ax, ds in zip(axes, DS):
    mr = rank[ds].mean().sort_values(); ax.set_xlim(1, k); ax.set_ylim(-1.5, 1.5); ax.grid(False)
    for s in ('left', 'top', 'right'): ax.spines[s].set_visible(False)
    ax.spines['bottom'].set_position(('data', 0)); ax.set_yticks([]); ax.set_xticks([])
    for r_ in range(1, k + 1): ax.plot([r_, r_], [0, .05], color=C['muted'], lw=.8); ax.text(r_, .08, str(r_), ha='center', va='bottom', fontsize=8, color=C['muted'])
    half = (k + 1) // 2
    for i, (m, r) in enumerate(mr.items()):
        left = i < half; lvl = -0.22 - 0.17 * (i if left else k - 1 - i); xe = 0.9 if left else k + 0.1
        ax.plot([r, r, xe], [0, lvl, lvl], color=C['muted'], lw=0.8, clip_on=False)
        ax.text(xe - 0.05 if left else xe + 0.05, lvl, f'{m} ({r:.1f})', ha='right' if left else 'left', va='center', fontsize=8, clip_on=False)
    cl, vals = [], mr.values
    for i in range(k):
        j = i
        while j + 1 < k and vals[j + 1] - vals[i] < cd: j += 1
        if j > i and not any(a <= i and j <= b for a, b in cl): cl.append((i, j))
    for n_, (i, j) in enumerate(cl): ax.plot([vals[i], vals[j]], [.38 + .13 * n_] * 2, color=C['blue'], lw=3, solid_capstyle='butt')
    ax.plot([1, 1 + cd], [1.35, 1.35], color=C['ink'], lw=1.5); ax.text(1 + cd / 2, 1.38, f'CD = {cd:.2f}', ha='center', va='bottom', fontsize=8)
    ax.set_title(f'{ds} - mean rank over {NF} folds (bars join models not distinguishable at 0.05, Nemenyi)', loc='left')
plt.subplots_adjust(left=.16, right=.84, hspace=.35); savefig('fig07_critical_difference')

mroc = 'Stacking' if 'Stacking' in MODELS else MODELS[-1]
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
for ds in DS:
    y = DS[ds]['ym']; p = np.zeros(len(y))
    for te, pr in oof[(ds, mroc)][:N_SPLITS]: p[te] = pr
    f, t, _ = roc_curve(y, p); pr_, rc_, _ = precision_recall_curve(y, p)
    ax[0].plot(f, t, color=DS_COL[ds], label=f'{ds} (AUC {roc_auc_score(y, p):.3f})'); ax[1].plot(rc_, pr_, color=DS_COL[ds], label=f'{ds} (base rate {y.mean():.2f})')
ax[0].plot([0, 1], [0, 1], ':', color=C['muted'], lw=1); ax[0].set(xlabel='False positive rate', ylabel='True positive rate', title=f'(a) ROC - {mroc}, pooled out-of-fold'); ax[0].legend()
ax[1].set(xlabel='Recall', ylabel='Precision', title='(b) Precision-recall', ylim=(0, 1.02)); ax[1].legend(loc='lower left')
savefig('fig06_roc_pr')

## C4: resampling factorial

In [ ]:
RS_CLFS = ['LR', 'DT', 'kNN', 'GNB', 'RF', 'HistGB']
if QUICK_MODE: RS_CLFS = ['LR', 'DT', 'GNB']
NF_RS = N_SPLITS * N_REPEATS_RESAMPLING
def rs_auc(ds, clfname, strat):
    est = make_clf(clfname, weighted=True) if strat == 'baseline' else InFoldResampler(strat, make_clf(clfname, weighted=False))
    return np.array([roc_auc_score(DS[ds]['ym'][te], p) for te, p in cv_probs(ds, est, N_REPEATS_RESAMPLING)])
rs = {}
for ds in DS:
    print(ds)
    for c in RS_CLFS:
        for s in ['baseline'] + list(RESAMPLERS):
            rs[(ds, c, s)] = cached(f'rs_{ds}_{c}_{s}', lambda: rs_auc(ds, c, s))
con = []
for ds in DS:
    for c in RS_CLFS:
        for s in RESAMPLERS:
            dlt = rs[(ds, c, s)] - rs[(ds, c, 'baseline')]
            p = 1.0 if (dlt == 0).all() else ss.wilcoxon(dlt).pvalue
            con.append(dict(Dataset=ds, Classifier=c, Strategy=s, dAUC_pp=100 * dlt.mean(), p=p, diffs=dlt))
con = pd.DataFrame(con); con['p_holm'] = holm(con.p.values)
con['outcome'] = np.where(con.p_holm >= .05, 'no effect', np.where(con.dAUC_pp > 0, 'win', 'loss'))
con.drop(columns='diffs').to_csv(TAB / 'table12b_all_contrasts.csv', index=False)
rows = []
for s, g in con.groupby('Strategy'):
    pooled = 100 * np.concatenate(g.diffs.values); lo, hi = boot_ci(pooled)
    rows.append({'Strategy': s, 'Wins': int((g.outcome == 'win').sum()), 'Losses': int((g.outcome == 'loss').sum()), 'No effect': int((g.outcome == 'no effect').sum()),
                 'Mean dAUC (pp)': g.dAUC_pp.mean(), 'CI low': lo, 'CI high': hi})
T12 = save_table(pd.DataFrame(rows), 'table12_resampling', f'TABLE 12 - resampling vs class-weighted baseline ({len(con)} contrasts, {NF_RS} folds, Holm at 0.05)', index=False, floatfmt=2)
allp = 100 * np.concatenate(con.diffs.values); lo, hi = boot_ci(allp)
print(f'\nPooled effect over all {len(con)} contrasts: {con.dAUC_pp.mean():+.2f} pp  (bootstrap 95% CI [{lo:+.2f}, {hi:+.2f}])')
print(f"Wins: {(con.outcome == 'win').sum()}   Losses: {(con.outcome == 'loss').sum()}   No effect: {(con.outcome == 'no effect').sum()}")
byds = con.pivot_table(index='Strategy', columns='Dataset', values='dAUC_pp', aggfunc='mean')[list(DS)]
byclf = con.pivot_table(index='Strategy', columns='Classifier', values='dAUC_pp', aggfunc='mean')[RS_CLFS]
save_table(byds, 'table12c_resampling_by_dataset', 'Mean dAUC (pp) by dataset', floatfmt=2)
save_table(byclf, 'table12d_resampling_by_classifier', 'Mean dAUC (pp) by classifier', floatfmt=2)
excl = con[con.Dataset != 'LAR-24K']; lo2, hi2 = boot_ci(100 * np.concatenate(excl.diffs.values))
print(f'Real datasets only: {excl.dAUC_pp.mean():+.2f} pp [{lo2:+.2f}, {hi2:+.2f}]; wins {(excl.outcome == "win").sum()}, losses {(excl.outcome == "loss").sum()}')
SUMMARY['C4_resampling'] = dict(table12=T12.to_dict('records'), pooled=con.dAUC_pp.mean(), ci=[lo, hi], wins=int((con.outcome == 'win').sum()),
                                losses=int((con.outcome == 'loss').sum()), n=len(con), real_only=dict(mean=excl.dAUC_pp.mean(), ci=[lo2, hi2]),
                                by_dataset=byds.to_dict(), by_classifier=byclf.to_dict(), winners=con[con.outcome == 'win'].drop(columns='diffs').to_dict('records'))

fig, ax = plt.subplots(1, 2, figsize=(12, 4)); yy = np.arange(len(T12))
for i, r in T12.iterrows():
    sig = r['CI high'] < 0 or r['CI low'] > 0
    ax[0].plot([r['CI low'], r['CI high']], [i, i], color=C['red'] if sig else C['muted'], lw=2)
    ax[0].plot(r['Mean dAUC (pp)'], i, 's' if sig else 'o', color=C['red'] if sig else C['muted'], ms=7, mfc=C['red'] if sig else 'white')
ax[0].axvline(0, color=C['ink'], lw=1, ls='--'); ax[0].set_yticks(yy, T12.Strategy); ax[0].invert_yaxis()
ax[0].set(xlabel='Δ ROC-AUC vs class-weighted baseline (pp)', title='(a) Pooled effect, bootstrap 95% CI (filled red = CI excludes 0)')
w = 0.26
for j, ds in enumerate(DS): ax[1].bar(np.arange(len(byds)) + (j - 1) * w, byds[ds], w - .02, color=DS_COL[ds], label=ds)
ax[1].axhline(0, color=C['ink'], lw=1); ax[1].set_xticks(np.arange(len(byds)), byds.index, rotation=30, ha='right'); ax[1].set(ylabel='Δ ROC-AUC (pp)', title='(b) By dataset'); ax[1].legend()
savefig('fig08_resampling')

## C4: calibration

In [ ]:
CAL_MODELS = ['LR', 'RF', 'HistGB']
def cal_unit(ds, mname):
    out = {}
    for lab, est in [('raw', make_clf(mname, weighted=False)),
                     ('platt', CalibratedClassifierCV(make_clf(mname, weighted=False), method='sigmoid', cv=3)),
                     ('isotonic', CalibratedClassifierCV(make_clf(mname, weighted=False), method='isotonic', cv=3))]:
        out[lab] = cv_probs(ds, est, N_REPEATS)
    return out
cal, rows = {}, []
for ds in DS:
    for m in CAL_MODELS:
        cal[(ds, m)] = cached(f'cal_{ds}_{m}', lambda: cal_unit(ds, m)); y = DS[ds]['ym']
        b = {lab: np.array([brier_score_loss(y[te], p) for te, p in r]) for lab, r in cal[(ds, m)].items()}
        a = {lab: np.mean([roc_auc_score(y[te], p) for te, p in r]) for lab, r in cal[(ds, m)].items()}
        d = b['platt'] - b['raw']; n = len(y); t, pnb = nadeau_bengio(b['platt'], b['raw'], n * .8, n * .2)
        rows.append({'Dataset': ds, 'Model': m, 'Brier raw': b['raw'].mean(), 'Platt': b['platt'].mean(), 'Isotonic': b['isotonic'].mean(),
                     'Platt change %': 100 * d.mean() / b['raw'].mean(), 'folds improved': f'{int((d < 0).sum())}/{len(d)}', 'NB p (Platt vs raw)': pnb,
                     'AUC raw': a['raw'], 'AUC Platt': a['platt']})
T14 = save_table(pd.DataFrame(rows), 'table14_calibration', 'TABLE 14 - calibration (Brier lower is better)', index=False)
SUMMARY['C4_calibration'] = T14.to_dict('records')
_real = T14[T14.Dataset != 'LAR-24K']
print('\nReal datasets only (use these as the headline; LAR-24K is a synthetic positive control):')
for _, r_ in _real.iterrows():
    print(f"  {r_['Dataset']} {r_['Model']:<7} Brier {r_['Brier raw']:.4f} -> {r_['Platt']:.4f}  ({r_['Platt change %']:+.1f}%, corrected p = {r_['NB p (Platt vs raw)']:.3f})")
print(f"  significant under the corrected test: {int((T14['NB p (Platt vs raw)'] < .05).sum())} of {len(T14)} dataset-model pairs")
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a_, ds in zip(ax, DS):
    y = DS[ds]['ym']
    for lab, col, mk in [('raw', C['red'], 'o'), ('platt', C['blue'], 's'), ('isotonic', C['aqua'], '^')]:
        r = cal[(ds, 'HistGB')][lab]; yy_, pp_ = np.concatenate([y[te] for te, _ in r]), np.concatenate([p for _, p in r])
        fr, mp = calibration_curve(yy_, pp_, n_bins=10, strategy='quantile'); a_.plot(mp, fr, marker=mk, color=col, ms=5, lw=1.5, label={'raw': 'Uncalibrated', 'platt': 'Platt', 'isotonic': 'Isotonic'}[lab])
    a_.plot([0, 1], [0, 1], ':', color=C['muted'], lw=1); t_ = T14[(T14.Dataset == ds) & (T14.Model == 'HistGB')].iloc[0]
    a_.set(xlabel='Mean predicted probability', ylabel='Observed frequency', title=f"{ds} - HistGB\nBrier {t_['Brier raw']:.4f} → {t_['Platt']:.4f} (Platt)")
ax[0].legend(); plt.tight_layout(); savefig('fig10_calibration')

## C4: cost-sensitive threshold (GC-1000, cost 5:1)

In [ ]:
C_BAD_ACCEPTED, C_GOOD_REJECTED = 5, 1
GRID = np.round(np.arange(0.01, 1.00, 0.01), 2)
def total_cost(y, p, t):
    acc_ = p >= t; return C_BAD_ACCEPTED * np.sum(acc_ & (y == 0)) + C_GOOD_REJECTED * np.sum(~acc_ & (y == 1))
def cal_hgb(): return CalibratedClassifierCV(HistGradientBoostingClassifier(random_state=SEED), method='sigmoid', cv=3)
def nested_fold(tr, te):
    v = DS['GC-1000']; X, y = v['Xm'], v['ym']
    pin = cross_val_predict(make_pipe('GC-1000', cal_hgb()), X.iloc[tr], y[tr], cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method='predict_proba')[:, 1]
    t_star = float(GRID[int(np.argmin([total_cost(y[tr], pin, t) for t in GRID]))])
    p = make_pipe('GC-1000', cal_hgb()).fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]
    return dict(n=len(te), t_star=t_star, cost_default=total_cost(y[te], p, .5), cost_nested=total_cost(y[te], p, t_star),
                cost_reject_all=total_cost(y[te], p, 1.01), cost_accept_all=total_cost(y[te], p, 0.0), te=te, p=p)
nest = cached('nested_threshold_gc', lambda: Parallel(n_jobs=N_JOBS)(delayed(nested_fold)(tr, te) for tr, te in get_folds('GC-1000', N_REPEATS)))
nd = pd.DataFrame([{k: v for k, v in r.items() if k not in ('te', 'p')} for r in nest])
red = lambda d: 100 * (1 - d.cost_nested.sum() / d.cost_default.sum())
rng = np.random.default_rng(SEED); boots = [red(nd.iloc[rng.integers(0, len(nd), len(nd))]) for _ in range(N_BOOT)]
ci = np.percentile(boots, [2.5, 97.5])
yg = DS['GC-1000']['ym']; pooled_p = np.concatenate([r['p'] for r in nest]); pooled_y = np.concatenate([yg[r['te']] for r in nest])
curve = np.array([total_cost(pooled_y, pooled_p, t) for t in GRID]) / N_REPEATS
t_pool = float(GRID[curve.argmin()]); c_def = total_cost(pooled_y, pooled_p, .5) / N_REPEATS
T15 = pd.DataFrame({'Quantity': ['Expected cost per applicant', 'Reduction vs default %', 'Bootstrap 95% CI on reduction', 'Selected threshold', 'Outer folds with lower cost',
                                 'Trivial policy: reject everyone (cost/applicant)', 'Trivial policy: accept everyone (cost/applicant)', 'Pooled-optimum threshold (in-sample)', 'Pooled-optimum reduction % (in-sample)'],
                    'Default t=0.50': [f'{nd.cost_default.sum() / nd.n.sum():.3f}', '-', '-', '0.50', '-', f'{nd.cost_reject_all.sum() / nd.n.sum():.3f}', f'{nd.cost_accept_all.sum() / nd.n.sum():.3f}', '-', '-'],
                    'Nested t*': [f'{nd.cost_nested.sum() / nd.n.sum():.3f}', f'{red(nd):.1f}', f'[{ci[0]:.1f}, {ci[1]:.1f}]', f'{nd.t_star.mean():.2f} (range {nd.t_star.min():.2f}–{nd.t_star.max():.2f})',
                                  f'{int((nd.cost_nested < nd.cost_default).sum())}/{len(nd)}', '', '', f'{t_pool:.2f}', f'{100 * (1 - curve.min() / c_def):.1f}']})
save_table(T15, 'table15_nested_threshold', 'TABLE 15 - nested cost-sensitive threshold selection, GC-1000, 5:1 cost matrix', index=False)
SUMMARY['C4_threshold'] = dict(cost_default=nd.cost_default.sum() / nd.n.sum(), cost_nested=nd.cost_nested.sum() / nd.n.sum(), reduction_pct=red(nd), ci=list(ci),
                               t_mean=nd.t_star.mean(), t_min=nd.t_star.min(), t_max=nd.t_star.max(), folds_lower=int((nd.cost_nested < nd.cost_default).sum()), folds=len(nd),
                               cost_reject_all=nd.cost_reject_all.sum() / nd.n.sum(), cost_accept_all=nd.cost_accept_all.sum() / nd.n.sum(), t_pooled=t_pool)
fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.axvspan(nd.t_star.min(), nd.t_star.max(), color=C['grid'], label='range of nested t*')
ax.plot(GRID, curve, color=C['blue'], label='expected cost (pooled out-of-fold)')
c_rej = 1000 * nd.cost_reject_all.sum() / nd.n.sum()
ax.axhline(c_rej, color=C['muted'], lw=1.2, ls=':', label=f'reject everyone (cost {c_rej:.0f})')
ax.plot(.5, c_def, 'o', color=C['ink'], mfc='white', ms=9, label=f'default t = 0.50 (cost {c_def:.0f})')
ax.plot(t_pool, curve.min(), 'D', color=C['red'], ms=8, label=f'pooled optimum t = {t_pool:.2f} (cost {curve.min():.0f})')
ax.set(xlabel='Acceptance threshold on P(good)', ylabel='Misclassification cost per 1,000 applicants', title='GC-1000, official UCI cost matrix (5:1)'); ax.legend(fontsize=8)
savefig('fig11_cost_curve')

## Permutation importance and learning curves

In [ ]:
def perm_unit(ds):
    v = DS[ds]; tr, te = train_test_split(np.arange(len(v['ym'])), test_size=0.3, stratify=v['ym'], random_state=SEED)
    m = make_pipe(ds, make_clf('RF')).fit(v['Xm'].iloc[tr], v['ym'][tr])
    r = permutation_importance(m, v['Xm'].iloc[te], v['ym'][te], scoring='roc_auc', n_repeats=5 if QUICK_MODE else 15, random_state=SEED, n_jobs=N_JOBS)
    return pd.DataFrame({'Attribute': v['Xm'].columns, 'dAUC': r.importances_mean, 'SD': r.importances_std}).sort_values('dAUC', ascending=False)
pi_ = {ds: cached(f'perm_{ds}', lambda: perm_unit(ds)) for ds in DS}
T13 = pd.concat([d.head(5).assign(Dataset=ds) for ds, d in pi_.items()])[['Dataset', 'Attribute', 'dAUC', 'SD']]
save_table(T13, 'table13_permutation_importance', 'TABLE 13 - top five attributes by permutation importance (held-out 30%, raw attributes, random forest)', index=False)
SUMMARY['perm_importance'] = T13.to_dict('records')
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
for a_, ds in zip(ax, DS):
    d = pi_[ds].head(6).iloc[::-1]; a_.barh(d.Attribute, d.dAUC, xerr=d.SD, color=DS_COL[ds], height=.6, error_kw=dict(lw=1, ecolor=C['muted']))
    a_.set(xlabel='Δ ROC-AUC when permuted', title=ds); a_.grid(axis='y', visible=False)
plt.tight_layout(); savefig('fig09_permutation_importance')

LC_MODELS = [m for m in ['LR', 'RF', 'Stacking'] if m in MODELS]; FRACS = [.2, .4, .6, .8, 1.0]
def lc_fold(ds, mname, tr, te, i):
    v = DS[ds]; rng = np.random.default_rng(SEED + i); out = []
    for f in FRACS:
        sub = tr if f == 1 else train_test_split(tr, train_size=f, stratify=v['ym'][tr], random_state=int(rng.integers(1e6)))[0]
        m = make_pipe(ds, make_clf(mname)).fit(v['Xm'].iloc[sub], v['ym'][sub]); out.append(roc_auc_score(v['ym'][te], m.predict_proba(v['Xm'].iloc[te])[:, 1]))
    return out
def lc_unit(ds, mname):
    return np.array(Parallel(n_jobs=N_JOBS)(delayed(lc_fold)(ds, mname, tr, te, i) for i, (tr, te) in enumerate(get_folds(ds, N_REPEATS))))
fig, ax = plt.subplots(1, 2, figsize=(11, 4)); lc_rows = []
for a_, ds in zip(ax, ['DHL-614', 'GC-1000']):
    for mname, col, mk in zip(LC_MODELS, [C['muted'], C['blue'], C['orange']], ['o', 's', '^']):
        r = cached(f'lc_{ds}_{mname}', lambda: lc_unit(ds, mname)); mu = r.mean(0); se = r.std(0, ddof=1) / math.sqrt(len(r))
        a_.plot(100 * np.array(FRACS), mu, marker=mk, color=col, label=mname, ms=5); a_.fill_between(100 * np.array(FRACS), mu - se, mu + se, color=col, alpha=.12, lw=0)
        lc_rows.append(dict(Dataset=ds, Model=mname, **{f'{int(100 * f)}%': m_ for f, m_ in zip(FRACS, mu)}))
    a_.set(xlabel='Training data used (%)', ylabel='ROC-AUC', title=f'{ds} (band = ±1 s.e. over {NF} folds)'); a_.legend()
savefig('fig12_learning_curves')
save_table(pd.DataFrame(lc_rows), 'fig12_learning_curves', 'Learning curves - mean ROC-AUC by training fraction', index=False)

## Stress tests

In [ ]:
T_THEORY = C_BAD_ACCEPTED / (C_BAD_ACCEPTED + C_GOOD_REJECTED)
N_APP = nd.n.sum()
pol = {'accept everyone': nd.cost_accept_all.sum() / N_APP, 'default t = 0.50': nd.cost_default.sum() / N_APP,
       'reject everyone': nd.cost_reject_all.sum() / N_APP,
       f'theoretical t = {T_THEORY:.3f} (no search)': sum(total_cost(yg[r['te']], r['p'], T_THEORY) for r in nest) / N_APP,
       'nested t* (manuscript method)': nd.cost_nested.sum() / N_APP}
best_trivial = min(pol['accept everyone'], pol['reject everyone'])
S1 = pd.DataFrame({'Cost per applicant': pol}); S1['vs default t=0.50 (%)'] = 100 * (S1['Cost per applicant'] / pol['default t = 0.50'] - 1)
S1['vs best trivial policy (%)'] = 100 * (S1['Cost per applicant'] / best_trivial - 1)
save_table(S1, 'stress1_threshold_policies', 'STRESS TEST 1 - GC-1000 cost per applicant under each policy (same out-of-fold predictions)', floatfmt=3)

LEAKERS = [('RandomOver', rs_random_over), ('SMOTE', rs_smote), ('ADASYN', rs_adasyn), ('SMOTETomek', rs_smote_tomek)]
def synth_leak(ds, s):
    v = DS[ds]; y = v['ym']; Z = make_pre(v['num'], v['cat']).fit_transform(v['Xm'])
    tr0, te0 = train_test_split(np.arange(len(y)), test_size=.2, stratify=y, random_state=s)
    out = {'none (clean split)': dict(clean=np.nan, leaky=np.nan, d2=d2_partition_overlap(pd.DataFrame(Z[tr0]), y[tr0], pd.DataFrame(Z[te0]), y[te0])['dup_fraction'],
                                      d2b=d2b_interpolation(Z[tr0], y[tr0], Z[te0], y[te0])['interpolated_fraction'], kdev=np.nan)}
    for name, fn in LEAKERS:
        Xr, yr = fn(Z[tr0], y[tr0], np.random.default_rng(s)); clean = accuracy_score(y[te0], RandomForestClassifier(random_state=s).fit(Xr, yr).predict(Z[te0]))
        Xo, yo = fn(Z, y, np.random.default_rng(s)); tr, te = train_test_split(np.arange(len(yo)), test_size=.2, stratify=yo, random_state=s)
        yh = RandomForestClassifier(random_state=s).fit(Xo[tr], yo[tr]).predict(Xo[te]); a = accuracy_score(yo[te], yh)
        out[name] = dict(clean=clean, leaky=a, d2=d2_partition_overlap(pd.DataFrame(Xo[tr]), yo[tr], pd.DataFrame(Xo[te]), yo[te])['dup_fraction'],
                         d2b=d2b_interpolation(Xo[tr], yo[tr], Xo[te], yo[te])['interpolated_fraction'], kdev=abs(cohen_kappa_score(yo[te], yh) - (2 * a - 1)))
    return out
s2rows = []
for ds in DS:
    sl = cached(f'stress_synth_leak_{ds}', lambda: Parallel(n_jobs=N_JOBS)(delayed(synth_leak)(ds, s) for s in range(N_LEAK_SPLITS)))
    for kname in ['none (clean split)'] + [n for n, _ in LEAKERS]:
        m = lambda f: float(np.mean([r[kname][f] for r in sl])); d2, d2b = 100 * m('d2'), 100 * m('d2b')
        s2rows.append({'Dataset': ds, 'Leak created by': kname, 'Leak-free acc': m('clean'), 'Leaky acc': m('leaky'), 'Inflation (pp)': 100 * (m('leaky') - m('clean')),
                       'D2 exact copies %': d2, 'D2 verdict': 'critical' if d2 > 0 else 'ok', 'D2b interpolated %': d2b, 'D2b verdict': 'critical' if d2b > 1 else 'ok',
                       'Caught by D2 or D2b': 'yes' if (d2 > 0 or d2b > 1) else 'no', 'D5 max |kappa-(2a-1)|': float(np.nanmax([r[kname]['kdev'] for r in sl])) if kname != 'none (clean split)' else np.nan})
S2 = save_table(pd.DataFrame(s2rows), 'stress2_leak_type_vs_detectors', f'STRESS TEST 2 - which detector sees which leak ({N_LEAK_SPLITS} splits per dataset; the clean split must stay "ok")', index=False, floatfmt=3)

TH_CLFS = [c for c in ['LR', 'RF', 'HistGB'] if c in MODELS]; TH_METRICS = ['ROC-AUC', 'Bal.acc', 'G-mean', 'MCC', 'F1', 'Brier']
TH_ARMS = ['unweighted', 'unweighted, t = base rate', 'class-weighted', 'RandomOver', 'SMOTE']
def th_unit(ds, c, arm):
    y = DS[ds]['ym']
    if arm.startswith('unweighted'):
        est, t = make_clf(c, weighted=False), (float(y.mean()) if 'base rate' in arm else 0.5)
    elif arm == 'class-weighted': est, t = make_clf(c, weighted=True), 0.5
    else: est, t = InFoldResampler(arm, make_clf(c, weighted=False)), 0.5
    return pd.DataFrame([all_metrics(y[te], p, t) for te, p in cv_probs(ds, est, N_REPEATS)])[TH_METRICS]
th = []
for ds in ['DHL-614', 'GC-1000']:
    for c in TH_CLFS:
        r = {arm: cached(f'stress_th2_{ds}_{c}_{arm}', lambda: th_unit(ds, c, arm)) for arm in TH_ARMS}; n = len(DS[ds]['ym'])
        for arm, df in r.items():
            row_ = dict(Dataset=ds, Classifier=c, Arm=arm, **df.mean().to_dict())
            for ref, lab in [('unweighted', 'NB p: G-mean vs unweighted'), ('class-weighted', 'NB p: G-mean vs class-weighted'), ('unweighted, t = base rate', 'NB p: G-mean vs threshold shift')]:
                row_[lab] = np.nan if arm == ref else nadeau_bengio(df['G-mean'], r[ref]['G-mean'], n * .8, n * .2)[1]
            th.append(row_)
S3 = save_table(pd.DataFrame(th), 'stress3_resampling_thresholded_metrics', f'STRESS TEST 3 - thresholded metrics, means over {NF} folds (t = 0.5 unless stated)', index=False, floatfmt=3)
_s3 = S3[S3.Arm.isin(['RandomOver', 'SMOTE'])]
print(f"\nResampling arms that beat a plain threshold shift on G-mean (corrected p < 0.05 and higher mean): "
      f"{int(sum((r_['NB p: G-mean vs threshold shift'] < .05) and (r_['G-mean'] > S3[(S3.Dataset == r_['Dataset']) & (S3.Classifier == r_['Classifier']) & (S3.Arm == 'unweighted, t = base rate')]['G-mean'].iloc[0]) for _, r_ in _s3.iterrows()))} of {len(_s3)}")
SUMMARY['stress'] = dict(threshold=S1.to_dict('index'), leak_vs_detectors=S2.to_dict('records'), thresholded=S3.to_dict('records'))

## Scorecard

In [ ]:
S = SUMMARY; lk = S['C1_leak']; t11 = {r['Dataset']: r for r in S['C3']['table11']}; calT = {(r['Dataset'], r['Model']): r for r in S['C4_calibration']}
t12 = {r['Strategy']: r for r in S['C4_resampling']['table12']}
def row(claim, item, paper, got, tol, fmt='{:.3f}'):
    ok = abs(got - paper) <= tol
    return {'Claim': claim, 'Quantity': item, 'Manuscript': fmt.format(paper), 'This run': fmt.format(got), 'Tolerance': tol, 'Reproduced': 'YES' if ok else 'NO'}
sc = [row('C1', 'LAR-24K rule mismatches', 0, S['C1_rule']['mismatches'], 0, '{:.0f}'),
      row('C1', 'Depth-4 tree acc with LTI', .9998, S['C1_rule']['tree_acc_with_lti'], .001, '{:.4f}'),
      row('C1', 'Depth-4 tree acc without LTI', .958, S['C1_rule']['tree_acc_without_lti'], .005),
      row('C1', 'DHL-614 leak inflation (pp)', 15.5, lk['DHL-614']['Inflation pp (mean)'], 2.5, '{:.1f}'),
      row('C1', 'GC-1000 leak inflation (pp)', 15.1, lk['GC-1000']['Inflation pp (mean)'], 2.5, '{:.1f}'),
      row('C1', 'LAR-24K leak inflation (pp)', 0.3, lk['LAR-24K']['Inflation pp (mean)'], 0.5, '{:.1f}'),
      row('C1', 'DHL-614 duplicated test rows %', 45.6, lk['DHL-614']['Duplicated test rows %'], 3, '{:.1f}'),
      row('C1', 'GC-1000 duplicated test rows %', 46.3, lk['GC-1000']['Duplicated test rows %'], 3, '{:.1f}'),
      row('C1', 'DHL-614 best 1-rule accuracy', .8094, S['C1_baselines']['DHL-614']['Best 1-rule acc'], .002, '{:.4f}'),
      row('C1', 'GC-1000 best 1-rule accuracy', .710, S['C1_baselines']['GC-1000']['Best 1-rule acc'], .002),
      row('C1', 'Transfer AUC (position-aligned)', .4974, S['C1_transfer']['position_aligned_auc'], .05),
      row('C2', 'D5 flag rate at pi=0.50', 1.0, S['C2_eval']['d5'][0]['flag_rate'], .02, '{:.2f}'),
      row('C2', 'D5 false-positive rate at pi=0.70', 0.0, S['C2_eval']['d5'][4]['flag_rate'], .01, '{:.3f}'),
      row('C3', 'DHL-614 best ROC-AUC', .7735, t11['DHL-614']['Best AUC'], .015),
      row('C3', 'GC-1000 best ROC-AUC', .7912, t11['GC-1000']['Best AUC'], .015),
      row('C3', 'LAR-24K best ROC-AUC', .9999, t11['LAR-24K']['Best AUC'], .001, '{:.4f}'),
      row('C3', 'Nemenyi CD (k=13, N=15)', 4.71, t11['DHL-614']['CD(0.05)'], .02, '{:.2f}'),
      row('C4', 'Resampling: significant wins', 0, S['C4_resampling']['wins'], 0, '{:.0f}'),
      row('C4', 'Resampling: significant losses', 29, S['C4_resampling']['losses'], 8, '{:.0f}'),
      row('C4', 'Resampling pooled dAUC (pp)', -0.67, S['C4_resampling']['pooled'], .35, '{:.2f}'),
      row('C4', 'NearMiss mean dAUC (pp)', -2.82, t12['NearMiss']['Mean dAUC (pp)'], 1.0, '{:.2f}'),
      row('C4', 'Platt Brier change %, LAR-24K HistGB', -27.5, calT[('LAR-24K', 'HistGB')]['Platt change %'], 8, '{:.1f}'),
      row('C4', 'Platt Brier change %, GC-1000 HistGB', -7.6, calT[('GC-1000', 'HistGB')]['Platt change %'], 3, '{:.1f}'),
      row('C4', 'Platt Brier change %, DHL-614 HistGB', -10.6, calT[('DHL-614', 'HistGB')]['Platt change %'], 3, '{:.1f}'),
      row('C4', 'Cost per applicant, default t=0.5', 1.031, S['C4_threshold']['cost_default'], .08),
      row('C4', 'Cost per applicant, nested t*', .561, S['C4_threshold']['cost_nested'], .04),
      row('C4', 'Nested cost reduction %', 45.6, S['C4_threshold']['reduction_pct'], 4, '{:.1f}'),
      row('C4', 'Mean nested threshold', .81, S['C4_threshold']['t_mean'], .04, '{:.2f}')]
for ds in DS:
    sc.append({'Claim': 'C3', 'Quantity': f'{ds}: best vs runner-up NOT significant (NB)', 'Manuscript': 'True', 'This run': str(t11[ds]['NB p'] >= .05), 'Tolerance': '', 'Reproduced': 'YES' if t11[ds]['NB p'] >= .05 else 'NO'})
SC = save_table(pd.DataFrame(sc), 'scorecard_vs_manuscript', 'REPRODUCTION SCORECARD - this run vs manuscript', index=False)
n_ok = int((SC.Reproduced == 'YES').sum()); print(f'\n{n_ok} of {len(SC)} checked quantities reproduced within tolerance.')
SUMMARY['scorecard'] = SC.to_dict('records')

rev = [dict(Where=f"{r_['Claim']} (scorecard)", Quantity=r_['Quantity'], Manuscript=r_['Manuscript'], Replace_with=r_['This run'],
            Action='REPLACE' if r_['Reproduced'] == 'NO' else 'keep (optionally update to this run)') for _, r_ in SC.iterrows()]
fa = S['C3']['friedman_across']; gnb = bench[('DHL-614', 'GNB')][METRICS].mean() if 'GNB' in MODELS else None
gs2 = S2[(S2.Dataset == 'GC-1000')].set_index('Leak created by')
rev += [dict(Where='VI-B', Quantity='Friedman across the 3 datasets, p', Manuscript='0.048', Replace_with=f"{fa['p']:.3f}", Action='REPLACE; say "not significant" if above 0.05'),
        dict(Where='VI-B', Quantity='Largest mean-rank difference vs CD (N=3)', Manuscript='10.0 vs 10.5', Replace_with=f"{fa['max_rank_diff']:.1f} vs {fa['cd']:.1f}", Action='REPLACE'),
        dict(Where='Table 6', Quantity='GC-1000 best one-attribute rule', Manuscript="0.710, credit_history != 'fully repaid'", Replace_with=f"{S['C1_baselines']['GC-1000']['Best 1-rule acc']:.3f}, {S['C1_baselines']['GC-1000']['Rule']}", Action='REPLACE'),
        dict(Where='VI-F / abstract', Quantity='Cost reduction vs reject-everyone policy (%)', Manuscript='not reported', Replace_with=f"{-S1.loc['nested t* (manuscript method)', 'vs best trivial policy (%)']:.1f}", Action='ADD next to the reduction vs t=0.5'),
        dict(Where='VI-F', Quantity='Cost per applicant: reject everyone', Manuscript='not reported', Replace_with=f"{pol['reject everyone']:.3f}", Action='ADD to Table 15'),
        dict(Where='VI-F', Quantity=f'Cost per applicant at theoretical t = {T_THEORY:.3f}', Manuscript='not reported', Replace_with=f"{[v_ for k_, v_ in pol.items() if k_.startswith('theoretical')][0]:.3f}", Action='ADD to Table 15'),
        dict(Where='V-D / Table 9', Quantity='SMOTE-before-split leak on GC-1000: inflation (pp) / D2 / D2b', Manuscript='not reported',
             Replace_with=f"{gs2.loc['SMOTE', 'Inflation (pp)']:.1f} pp / {gs2.loc['SMOTE', 'D2 exact copies %']:.1f}% / {gs2.loc['SMOTE', 'D2b interpolated %']:.1f}%", Action='ADD; state that D2 sees exact copies only')]
if gnb is not None:
    rev.append(dict(Where='Table 17', Quantity='GNB on DHL-614: F1 / Brier', Manuscript='0.0154 / 0.6810', Replace_with=f"{gnb['F1']:.4f} / {gnb['Brier']:.4f}", Action='REPLACE (manuscript row looks like a bug)'))
for _, r_ in T14[T14.Dataset != 'LAR-24K'].iterrows():
    rev.append(dict(Where='VI-E / abstract', Quantity=f"Platt Brier change, {r_['Dataset']} {r_['Model']}", Manuscript='see Table 14', Replace_with=f"{r_['Platt change %']:+.1f}% (corrected p = {r_['NB p (Platt vs raw)']:.3f})", Action='Lead with real-data values, not the LAR-24K maximum'))
REV = save_table(pd.DataFrame(rev), 'numbers_for_manuscript', 'NUMBERS FOR THE MANUSCRIPT - what to replace or add', index=False)
with open(RES / 'methods_details.md', 'w') as f:
    f.write(f"""# Details to state in the methods section (generated from this run)

- Software: Python {platform.python_version()}, scikit-learn {sklearn.__version__}, numpy {np.__version__}, pandas {pd.__version__}, scipy {scipy.__version__}. Master seed {SEED}.
- Benchmark: stratified {N_SPLITS}-fold cross-validation repeated {N_REPEATS} times ({NF} folds). LAR-24K classifier panel: fixed stratified subsample of {LAR_SUBSAMPLE} rows.
- Classifiers ({len(MODELS)}): {', '.join(MODELS)}. Library defaults; class_weight='balanced' where the learner supports it (LR, SVM, DT, RF, ExtraTrees, HistGB); MLP max_iter=500.
- Leakage experiment (Table 5): {N_LEAK_SPLITS} independent stratified 80/20 splits (seeds 0..{N_LEAK_SPLITS - 1}); random oversampling of the minority class to exact balance; random forest with default settings.
  A duplicated row is a validation row whose predictors and label are identical to at least one training row.
- Resampling factorial (Table 12): classifiers {', '.join(RS_CLFS)}; strategies {', '.join(RESAMPLERS)}; {NF_RS} folds; baseline is the class-weighted learner, resampled arms use the same learner without class weights.
  SMOTE and ADASYN use k = 5 neighbours; NearMiss is version 1 with 3 neighbours; SMOTE-Tomek removes both members of each Tomek link. Samplers act on the encoded training fold.
  Significance: two-sided Wilcoxon signed-rank on paired fold ROC-AUC, Holm-corrected across all {len(con)} contrasts.
- D5 simulation: tables of 11 rows, validation size 200, classifier sensitivity and specificity drawn uniformly from [0.55, 0.98], scores rounded to two decimals; flag when at least 80% of rows satisfy |kappa - (2*accuracy - 1)| <= 0.025.
- D2b (interpolation check): a validation row is flagged if it lies on the segment between two of its 10 nearest same-class training rows (relative tolerance 1e-6); critical when more than 1% of validation rows are flagged.
- Calibration: Platt and isotonic via CalibratedClassifierCV with internal 3-fold cross-validation inside each training fold; uncalibrated models are fitted without class weights.
- Threshold selection: calibrated HistGB (Platt); threshold grid 0.01..0.99 step 0.01; inner 5-fold out-of-fold predictions; cost matrix bad-accepted = {C_BAD_ACCEPTED}, good-rejected = {C_GOOD_REJECTED}.
- Permutation importance: random forest, 30% hold-out, 15 repeats, ROC-AUC scoring, computed on the raw attributes.
""")
print('[saved] methods_details.md')
SUMMARY['env'] = dict(python=platform.python_version(), sklearn=sklearn.__version__, numpy=np.__version__, pandas=pd.__version__, scipy=scipy.__version__, quick=QUICK_MODE, seed=SEED)
def _j(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)
json.dump(SUMMARY, open(RES / 'results_summary.json', 'w'), indent=1, default=_j)
with open(RES / 'REPORT.md', 'w') as f:
    f.write(f'# Run report\n\nQuick mode: {QUICK_MODE}. scikit-learn {sklearn.__version__}. Seed {SEED}.\n\n## Scorecard vs manuscript ({n_ok}/{len(SC)} reproduced)\n\n')
    f.write('| ' + ' | '.join(SC.columns) + ' |\n|' + '---|' * len(SC.columns) + '\n')
    for _, r in SC.iterrows(): f.write('| ' + ' | '.join(str(x) for x in r.values) + ' |\n')
    f.write('\nAll tables are in `tables/`, all figures in `figures/`, every headline number in `results_summary.json`.\n')
print(f'\nAll outputs are in: {RES}')
for sub in (TAB, FIG): print(f'  {sub.name}/: {len(list(sub.iterdir()))} files')